In [13]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate
import copy
from numpy import unique
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Embedding
from keras.layers import concatenate
from keras.utils import plot_model
import math
from tensorflow.keras.layers import Reshape
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import LSTM
from transformers import XLMRobertaTokenizer, TFAutoModel
from transformers import TFAutoModelForSequenceClassification
import matplotlib.pyplot as plt
#from keras.utils import plot_model
import os
import torch
from huggingface_hub import from_pretrained_keras
from collections import Counter
import tensorflow as tf
from tensorflow.keras import optimizers
from tensorflow.keras import metrics
from tensorflow.keras import losses
from transformers import pipeline
from keras.layers import Softmax
from tensorflow.keras.utils import plot_model
from tensorflow.keras import regularizers
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import category_encoders as ce
from tensorflow.keras.optimizers.schedules import PolynomialDecay
from sklearn.metrics import mean_squared_error

In [2]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
#let's filter on the outliers and see how model performance increases
def filter_outliers(df, upper, lower, target_col = "award_contract/val_total: 0"):
    """This function only works on numerical columns"""
    data_array = np.sort(df[target_col].to_numpy())
    cut_off_low_indice = math.floor(lower * len(data_array))
    cut_off_high_indice = math.floor(upper * len(data_array)) - 1
    low_amount = data_array[cut_off_low_indice]
    high_amount = data_array[cut_off_high_indice]

    outlier_indices = []

    for i in range(0, len(df)):
        if df[target_col].iloc[i] > high_amount:
            outlier_indices.append(i)
        elif df[target_col].iloc[i] < low_amount:
            outlier_indices.append(i)
        else:
            continue

    print("{} rows were dropped because of outliers".format(len(outlier_indices)))
    df = df.drop(labels = outlier_indices)

def merge_text_cols(df):
  text_columns = ["short_descr", "ac criteria", "object_contract/title", "object descr titles", "ac cost criteria"]
  merged_text = []
  for i in range(0, len(df)):
    text_row = ""
    for col in text_cols:
      text_cell = str(df[col].iloc[i])
      if len(text_cell) > 0 and text_cell not in text_row:
        text_row += text_cell
    if len(text_row) > 0:
      merged_text.append(text_row)
    else:
      merged_text.append(np.nan)
  df["merged text"] = merged_text
  df = df.drop(columns = text_cols)
  return df

def map_target_var(award_price_divisions, y):
  y_encoded = []

  for i in range(len(y)):
    for j, interval in enumerate(award_price_divisions.keys()):
      lower_bound = award_price_divisions[interval][0]
      upper_bound = award_price_divisions[interval][1]

      if y[i] >= lower_bound and y[i] < upper_bound:
        y_encoded.append(j)
      else:
        continue

  #check for remaining values at the higher end of the array
  left_over_cases = len(y) - len(y_encoded)
  if left_over_cases > 0:
    for i in range(0, left_over_cases):
      y_encoded.append(j)

  y_encoded = np.array(y_encoded)

  return y_encoded

#we need to get a better feeling for the distribution accros award price "classes", is there a distribution to be found which could improve accuracy, propagating to the final model in MAE
def award_price_distribution(y, amount_of_classes: int, make_labels):
  if make_labels == True:
    award_price_divisions = {}
    y_sorted = np.sort(y)
    length_y = len(y_sorted)
    length_class = int(length_y / amount_of_classes)
    lower_index = 0
    for i in range(1, amount_of_classes):
      upper_index = length_class * i
      lower_value = y_sorted[lower_index]
      upper_value = y_sorted[upper_index]
      print("lower index: {}, lower value {}, upper index {}, upper value {}".format(lower_index, lower_value, upper_index, upper_value))
      award_price_divisions[str(lower_value) + "-" + str(upper_value)] = [lower_value, upper_value]
      print("length of class = {}".format(len(y_sorted[lower_index:upper_index])))
      lower_index += 1 * length_class

    y = map_target_var(award_price_divisions, y)
    return y
  else:
    print("no encoding or labels were made for the target variable")
    return y
  
def prepare_data(df, train_or_test, amount_of_classes, drop_columns, upper_limit_outliers = 1, lower_limit_outliers = 0, make_labels = False):
    
  df = df.drop(columns = drop_columns)
  #remove outliers
  if train_or_test == "train":
    filter_outliers(df, upper = upper_limit_outliers, lower = lower_limit_outliers)
  else:
    print("no outliers were removed because it is the test set")

  df = merge_text_cols(df)

  #delete rows which have no text
  rows_no_text = []
  for i in range(0, len(df)):
    if pd.isna(df["merged text"].iloc[i]) == True:
      rows_no_text.append(i)

  df = df.drop(labels = rows_no_text)
  print("{} rows were dropped because they contained no text".format(len(rows_no_text)))

  #split the input and output columns
  target_column = "award_contract/val_total: 0"
  input_cols = [col for col in df if col != target_column and col != "merged text"]
  X_num_cat = df[input_cols].values
  X_text = df["merged text"].values.tolist()
  y = df[target_column].astype(int).values
  y_encoded = award_price_distribution(y, amount_of_classes, make_labels)

  return (X_num_cat, X_text, y, y_encoded)

def plot_metrics(ledger, save_path = None):
  #model results is a dict from a keras object
  model_results = ledger.history
  plt.figure(figsize=(10, 10))

  # Create the top row of subplots
  ax1 = plt.subplot(3, 1, 1)
  ax2 = plt.subplot(3, 1, 2)
  ax3 = plt.subplot(3, 1, 3)
  axes = [ax1, ax2, ax3]
  for i, key in enumerate(list(model_results.keys())[:int(0.5*len(model_results.keys()))]):
    loss_train = model_results[key]
    loss_val   = model_results["val_" + key]
    epochs = range(0,len(loss_train))

    axes[i].plot(epochs, loss_train, "g", label = "Training {}".format(key))
    axes[i].plot(epochs, loss_val, "b", label = "Validation {}".format("val_" + key))
    axes[i].title.set_text("Training and validation {}".format(key))
    axes[i].set_xlabel("Epochs")
    axes[i].set_ylabel("{}".format(key))
    axes[i].legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, format='png')

In [4]:
def train_validate_test_split(df, train, validate):
    target_col = "award_contract/val_total: 0"
    sort_col = "date_conclusion_contract"
    text_cols = ["short_descr", "ac criteria", "object_contract/title", "object descr titles", "ac cost criteria"]

    df = df.sort_values(by = [sort_col], axis = 0, ascending = True)
    df = df.drop(columns = ([sort_col] + text_cols))
    train_indice = int(train * len(df))
    validate_indice = int(validate * len(df)) + train_indice

    train_set = df[:train_indice-1]
    val_set = df[train_indice:validate_indice-1]
    test_set = df[validate_indice:]

    X_train = train_set.drop(columns = [target_col]).astype(float).values
    y_train = train_set[target_col].astype(float).values

    X_val = val_set.drop(columns = [target_col]).astype(float).values
    y_val = val_set[target_col].astype(float).values

    X_test = test_set.drop(columns = [target_col]).astype(float).values
    y_test = test_set[target_col].astype(float).values

    return  X_train, y_train, X_val, y_val, X_test, y_test

notes on model architecture
initial two runs were with 32-8-32 <br>
new run form 19:20 onwards 32-8-32-8-32

    #define the layers
    input_num_cat = Input(shape=input_dimension)
    x = Dense(256, activation = "relu")(input_num_cat) #kernel_regularizer=regularizers.L1L2(l1=0.005)
    x = Dropout(rate = 0.2)(x)
    x = Dense(128, activation = "relu")(x)
    x = Dropout(rate=0.1)(x)
    x = Dense(32, activation = "relu")(x)
    x = Dropout(rate=0.1)(x)
    x = Dense(64, activation = "relu")(x)
    x = Dropout(rate=0.1)(x)
    x = Dense(32, activation = "relu")(x)

In [79]:
def create_model(input_dimension, X_train, y_train, X_val, y_val, X_test, y_test, start_learning_rate, end_learning_rate, activation_function, loss_function, batch_size, epochs = 50):
    #let's play around a little more with the keras model
    metrics = ["mse", "mae", "R2Score"]

    #define the layers
    input_num_cat = Input(shape=input_dimension)
    x = Dense(256, activation = "relu")(input_num_cat) #kernel_regularizer=regularizers.L1L2(l1=0.005)
    x = Dropout(rate = 0.2)(x)
    x = Dense(128, activation = "relu")(x)
    x = Dropout(rate=0.1)(x)
    x = Dense(32, activation = "relu")(x)
    x = Dropout(rate=0.1)(x)
    x = Dense(64, activation = "relu")(x)
    x = Dropout(rate=0.1)(x)
    x = Dense(32, activation = "relu")(x)
    regression_layer = Dense(1, activation="linear")(x)
    model_num_cat = Model(inputs = [input_num_cat],
                          outputs = regression_layer)

    num_train_steps = len(X_train) / batch_size * epochs
    lr_scheduler = PolynomialDecay(
        initial_learning_rate=start_learning_rate, end_learning_rate=end_learning_rate, decay_steps=num_train_steps
    )

    checkpoint_path = "Results/Models/"
    cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path, 
                                                     verbose=1)

    optimizer = Adam(learning_rate=lr_scheduler)
    num_train_steps

    model_num_cat.compile(loss=loss_function,
                          optimizer = optimizer,
                          metrics = metrics)
    model_num_cat.summary()

    history = model_num_cat.fit(x = [X_train], y=y_train,
                                  validation_data = (X_val, y_val),
                                  epochs = epochs,
                                  batch_size = batch_size,
                                  callbacks=[cp_callback])

    y_pred = model_num_cat.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    
    metric = tf.keras.metrics.R2Score()
    metric.update_state(y_test.reshape(-1,1), y_pred)
    r2_result = metric.result().numpy()
    
    return model_num_cat, history, mae, r2_result, mse

In [45]:
#load the data
df_train = pd.read_pickle("Data/df_preprocessed_train_text.pickle").reset_index(drop = True)
df_test = pd.read_pickle("Data/df_preprocessed_test_text.pickle").reset_index(drop = True)
df_total = pd.concat([df_train, df_test]).reset_index(drop=True)

In [ ]:
loss_functions = ["mae", "mse"]
activation_functions= ["relu", "selu"]
results = {}
for loss_function in loss_functions:
    for activation_function in activation_functions:
        #DEFINE THE MODEL AND TRAIN ON THE WHOLE DATASET
        X_train, y_train, X_val, y_val, X_test, y_test = train_validate_test_split(df_total, 0.6, 0.2)

        #COMPILE AND TRAIN THE MODEL
        model, history, mae, r2_result, mse = create_model(input_dimension=X_train.shape[1], 
                     X_train = X_train, 
                     y_train = y_train, 
                     X_val = X_val, 
                     y_val = y_val, 
                     X_test = X_test, 
                     y_test = y_test,
                     start_learning_rate= 0.01,
                     end_learning_rate = 0.0001,
                     epochs = 200,
                     batch_size = 32,
                     loss_function = loss_function,
                     activation_function = activation_function)
        results["{}: {}".format(loss_function, activation_function)] = {"history": history, 
                                                                        "mae": mae,
                                                                        "mse": mse,
                                                                        "r2": r2_result}

In [ ]:
results

In [59]:
#take variables out iteratively to get model results in steps
cpv_cols = [col for col in df_total.columns if "cpv_code" in col]
country_cols = [col for col in df_total.columns if "country" in col]
type_contract_cols = [col for col in df_total.columns if "type_contract" in col]
ca_type_cols = [col for col in df_total.columns if "ca_type" in col]
language = [col for col in df_total.columns if "language" in col]

leave_one_out_cols = {"cpv_cols": cpv_cols,
                      "country_cols":country_cols,
                      "type_contract_cols":type_contract_cols,
                      "ca_type_cols": ca_type_cols,
                      "language":language,
                      }

In [ ]:
#DEFINE THE MODEL AND TRAIN ON THE WHOLE DATASET
X_train, y_train, X_val, y_val, X_test, y_test = train_validate_test_split(df_total, 0.6, 0.2)

#COMPILE AND TRAIN THE MODEL
model, history, mae, r2_result = create_model(input_dimension=X_train.shape[1], 
             X_train = X_train, 
             y_train = y_train, 
             X_val = X_val, 
             y_val = y_val, 
             X_test = X_test, 
             y_test = y_test,
             start_learning_rate= 0.05,
             end_learning_rate = 0.00001,
             epochs = 200,
             batch_size = 32,
             activation_function="relu",
             loss_function="mae")

In [ ]:
results = {}

for i, feature in enumerate(leave_one_out_cols.keys()):
    df_subset = copy.deepcopy(df_total.drop(columns = leave_one_out_cols[feature], axis=0))
    
    print(i, feature)

    #DEFINE THE MODEL AND TRAIN ON THE WHOLE DATASET
    X_train, y_train, X_val, y_val, X_test, y_test = train_validate_test_split(df_subset, 0.6, 0.2)

    #COMPILE AND TRAIN THE MODEL
    model, history, mae, r2_result, mse = create_model(input_dimension=X_train.shape[1], 
                 X_train = X_train, 
                 y_train = y_train, 
                 X_val = X_val, 
                 y_val = y_val, 
                 X_test = X_test, 
                 y_test = y_test,
                 start_learning_rate= 0.05,
                 end_learning_rate = 0.0001,
                 epochs = 200,
                 batch_size = 32,
                 loss_function = "mae",
                 activation_function = "relu")
    results[feature] = {"history": history, 
                        "mae": mae,
                        "mse": mse,
                        "r2": r2_result}

In [81]:
#SAVE MODEL RESULTS
#with open("Results\Model results\one_in_one_out_results", "wb") as f:
#    pickle.dump(results, f)

In [75]:
with open("Results\Model results\one_in_one_out_results", "rb") as f:
    results = pickle.load(f)

In [ ]:
results

In [60]:
#store the model
model.save("Results/Models/test_model.keras")

In [61]:
num_cat_model = tf.keras.models.load_model("Results/Models/test_model.keras")

In [ ]:
num_cat_model.summary()

In [ ]:
plot_metrics(results["history"])

In [ ]:
#model should be saved in history
history.history

In [ ]:
results = {}

for i, feature in enumerate(leave_one_out_cols.keys()):
    df_subset = copy.deepcopy(df_total.drop(columns = leave_one_out_cols[feature], axis=0))
    
    print(i, feature)

    X_train, y_train, X_val, y_val, X_test, y_test = train_validate_test_split(df_subset, 0.6, 0.2)
    input_dimension = X_train.shape[1]
    
    history, mae, r2_result = create_model(input_dimension=input_dimension, 
             X_train = X_train, 
             y_train = y_train, 
             X_val = X_val, 
             y_val = y_val, 
             X_test = X_test, 
             y_test = y_test,
             learning_rate= 0.05,
             epochs = 100)
    
    results["without {}".format(feature)] = {"history": history,
                                             "mae_test": mae,
                                             "R2_test": r2_result}

In [73]:
def plot_metrics(results, save_path=None):
    plt.figure(figsize=(15, 25))

    axes = []
    for i in range(1, 20):
        ax = plt.subplot(5, 4, i)
        axes.append(ax)

    for i, subset in enumerate(results.keys()):
        model_results = results[subset]["history"].history
        for j, key in enumerate(list(model_results.keys())[:int(0.5 * len(model_results.keys()))]):
            loss_train = model_results[key]
            loss_val = model_results["val_" + key]
            epochs = range(0, len(loss_train))

            axes[i * 3 + j].plot(epochs, loss_train, "g", label="Training {}".format(key))
            axes[i * 3 + j].plot(epochs, loss_val, "b", label="Validation {}".format("val_" + key))
            axes[i * 3 + j].set_title("Training and validation of {} with metric {}".format(subset, key), fontsize = 10)
            axes[i * 3 + j].set_xlabel("Epochs")
            axes[i * 3 + j].set_ylabel("{}".format(key))
            plt.tight_layout()
    
    handles, labels = axes[0].get_legend_handles_labels()
    plt.legend(handles, labels, bbox_to_anchor=(1, 6), loc='upper left')

    if save_path:
        plt.savefig(save_path, format='png')
    else:
        plt.show()

In [48]:
#with open("Figures\Model results\model_num_cat_14_11.19:18", "wb") as f:
#    pickle.dump(results, f)

In [ ]:
results

In [ ]:
plot_metrics(results)


-----------------------------------------------------------
EVERYTHING BELOW THIS IS OLD
-----------------------------------------------------------
-----------------------------------------------------------

In [ ]:
amount_of_classes = 7

X_train_num_cat, X_train_text, y_train, y_train_encoded = prepare_data(df_train,
                                                      upper_limit_outliers = 0.95,
                                                      lower_limit_outliers = 0.05,
                                                      train_or_test = "train",
                                                      amount_of_classes = amount_of_classes,
                                                      make_labels = True)

X_test_num_cat, X_test_text, y_test, y_test_encoded = prepare_data(df_test,
                                                   train_or_test = "test",
                                                   amount_of_classes = amount_of_classes,
                                                   make_labels = True)

In [6]:
subset_train = len(df_train) #or something else
subset_test = len(df_test) #or something else

#slice the target variable
y_train = y_train[:subset_train]
y_test = y_test[:subset_test]

#slice the text features
X_train_text = X_train_text[:subset_train]
X_test_text = X_test_text[:subset_test]

#slice the num_cat features
X_train_num_cat = X_train_num_cat[:subset_train]
X_test_num_cat = X_test_num_cat[:subset_test]

In [ ]:
y_train

In [ ]:
#let's play around a little more with the keras model
optimizer = keras.optimizers.Adam(learning_rate=0.0001) #tf.keras.optimizers.experimental.Adagrad(learning_rate=0.001)
loss_function = "msle"
metrics = ["mae", "mse", "msle"]
#define the layers
input_dim_num_cat = 55 #len(X_train[0])
input_num_cat = Input(shape=input_dim_num_cat)
x = Dense(32, activation = "relu",  kernel_regularizer=regularizers.L1L2(l1=0.1))(input_num_cat)
x = Dropout(rate = 0.4)(x)
x = Dense(8, activation = "relu", kernel_regularizer=regularizers.L1L2(l1=0.1))(x)
x = Dropout(rate=0.3)(x)
x = Dense(32, activation = "relu", kernel_regularizer=regularizers.L1L2(l2=0.1))(x)
regression_layer = Dense(1, activation="linear")(x)
model_num_cat = Model(inputs = [input_num_cat],
                      outputs = regression_layer)

model_num_cat.compile(loss=loss_function,
              optimizer = optimizer,
              metrics = metrics)
model_num_cat.summary()

history = model_num_cat.fit(x = [X_train_num_cat], y=y_train,
          validation_data = (X_test_num_cat, y_test),
          epochs = 100,
          batch_size = 32)

In [38]:
with open("Data/history_num_cat_model_A_msle.pickle", "wb") as f:
    pickle.dump(history, f)

In [ ]:
keras.utils.plot_model(model_num_cat, show_shapes=True)

In [40]:
def plot_metrics_single(ledger, save_path = None):
  #model results is a dict from a keras object
  model_results = ledger.history
  plt.figure(figsize=(10, 10))

  # Create the top row of subplots
  ax1 = plt.subplot(3, 1, 1)
  ax2 = plt.subplot(3, 1, 2)
  ax3 = plt.subplot(3, 1, 3)
  axes = [ax1, ax2, ax3]
  for i, key in enumerate(list(model_results.keys())[:int(0.5*len(model_results.keys()))]):
    loss_train = model_results[key]
    loss_val   = model_results["val_" + key]
    epochs = range(0,len(loss_train))

    axes[i].plot(epochs, loss_train, "g", label = "Training {}".format(key))
    axes[i].plot(epochs, loss_val, "b", label = "Validation {}".format("val_" + key))
    axes[i].title.set_text("Training and validation {}".format(key))
    axes[i].set_xlabel("Epochs")
    axes[i].set_ylabel("{}".format(key))
    axes[i].legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, format='png')

In [ ]:
plot_metrics(history)

RUN THE ALGORITHM ON A SUBSET OF THE DATA AND SEE WHAT COMES OUT

In [ ]:
with open("Results/Model results/history_final_model.pickle", "rb") as f:
    history = pickle.load(f)

In [50]:
#let's retrieve the results of each model
with open("Results/Model results/model_num_cat_results_32_8_32", "rb") as f:
    model_results_A = pickle.load(f)

with open("Results/Model results/model_num_cat_results_32-16-8-16-32", "rb") as f:
    model_results_B = pickle.load(f)

with open("Results/Model results/model_num_cat_results_256-128-32-64-32", "rb") as f:
    model_results_C= pickle.load(f)

In [ ]:
#retrieve the scores for the test set
model_results_A

In [ ]:
model_results_B

In [ ]:
model_results_C

In [ ]:
3696266057656

In [ ]:
plot_metrics(model_results_C["mae: relu"]["history"])